# convert

In [ ]:
///- --test static

open testing

## convert

### unativeint

In [ ]:
type unativeint_switch =
    {
        Gleam : $'unativeint'
        Lua : $'unativeint'
        Fsharp : $'unativeint'
        Python : $'unativeint'
        TypeScript : $'unativeint'
        Cpp : $'unativeint'
        CppHost : $'unativeint'
        Delphi : $'unativeint'
        Cuda : $'unativeint'
        Rust : $'usize'
        Zig : $'usize'
    }
nominal unativeint = $'backend_switch `(unativeint_switch)'

### to'

In [ ]:
inl to' forall t u. (fn : t -> option u) (x : t) : option u =
    real
        typecase t * u with
        | ~x * ~x => Some `u x
        | int * int => Some `u (conv `t `u x)
        | float * float => Some `u (conv `t `u x)
        | int * float => Some `u (conv `t `u x)
        | float * int => Some `u (conv `t `u x)
        | i64 * f64 => Some `u (conv `t `u x)
        | f64 * i64 => Some `u (conv `t `u x)
        | i32 * f64 => Some `u (conv `t `u x)
        | f64 * i32 => Some `u (conv `t `u x)
        | i64 * f32 => Some `u (conv `t `u x)
        | f32 * i64 => Some `u (conv `t `u x)
        | char * bool => Some `u (x = '1')
        | bool * char => Some `u (if x then '1' else '0')
        | bool * _ => Some `u (if x then 1 else 0)
        | _ => fn x

### to

In [ ]:
inl to forall t u. (x : t) : u =
    if is_rust ()
    then
        inl from_string : bool =
            real
                typecase t with
                | string => true
                | _ => false
        x
        |> to' fun x =>
            inl y : u =
                if from_string
                then $'(!x).parse::<`u>().unwrap()'
                else $'(!x as `u)'
            Some y
        |> optionm.value
    else
        x
        |> to' fun x => $'!x ' |> Some
        |> optionm.value

inl numeric_conv forall t u. (x : t) : u =
    if (real typecase t with i8 => true | i16 => true | i32 => true | i64 => true | u8 => true | u16 => true | u32 => true | u64 => true | f32 => true | f64 => true | _ => false)
    then real conv `t `u x
    else $'!x ' : u

### convert_i32_base

In [ ]:
inl convert_i32_base forall t. (base : i32) (x : t) : i32 =
    backend_switch {
        Gleam = fun () =>
            global "import gleam/int"
            inl (n : int), (err : int) =
                $'case int.base_parse(!x, !base) { Ok(n) -> ##(n, 0) Error(Nil) -> ##(0, 1) }'
            if err = 0
            then n |> Some
            else None
            |> optionm.value
        Lua = fun () => $'tonumber( !x, !base )' : i32
        Fsharp = fun () => $'System.Convert.ToInt32 (!x, !base)' : i32
        Python = fun () => $'int (!x, !base)' : i32
        Rust = fun () => $'i32::from_str_radix(&*(!x), (!base) as u32).unwrap()' : i32
        TypeScript = fun () => $'parseInt(!x, !base)' : i32
        C = fun () => $'(int32_t)strtol((!x)->ptr, NULL, !base)' : i32
        Delphi = fun () =>
            if base = 16 then $'StrToInt(##36 + !x)' : i32
            elif base = 2 then $'StrToInt(##37 + !x)' : i32
            elif base = 8 then $'StrToInt(##38 + !x)' : i32
            elif base = 10 then $'StrToInt(!x)' : i32
            else $'backend_switch_has_no_Delphi_arm_in_lib_spiral' : i32
        Wasm = fun () => $'(i32.wrap_i64 (call $parse_i64 !x !base))' : i32
        Zig = fun () => $'(std.fmt.parseInt(i32, std.mem.trim(u8, !x, " \\t\\r\\n"), \@intCast(!base)) catch 0)' : i32
    }

### convert_i32

In [ ]:
inl is_string_type forall t. (_ : t) : bool = real typecase t with string => true | _ => false
inl is_float_type forall t. (_ : t) : bool = real typecase t with f64 => true | f32 => true | _ => false

inl convert_i32 forall t. (x : t) : int =
    backend_switch {
        Gleam = fun () => x |> convert_i32_base 10 : int
        Lua = fun () => x |> convert_i32_base 10 : int
        Fsharp = fun () => $'System.Convert.ToInt32(!x)' : int
        Python = fun () => x |> convert_i32_base 10 : int
        Rust = fun () =>
            if (real typecase t with string => true | _ => false)
            then x |> convert_i32_base 10 : int
            elif (real typecase t with f64 => true | f32 => true | _ => false)
            then $'((!x).round_ties_even() as i32)' : int
            else $'(!x as i32)' : int
        C = fun () =>
            if is_string_type x
            then x |> convert_i32_base 10 : int
            elif is_float_type x
            then
                global "#include <math.h>"
                $'(int32_t)nearbyint(!x)' : int
            else $'(int32_t)(!x)' : int
        Delphi = fun () =>
            if is_string_type x
            then x |> convert_i32_base 10 : int
            elif is_float_type x
            then $'Round(!x)' : int
            else $'Int32(!x)' : int
        TypeScript = fun () =>
            if is_string_type x
            then x |> convert_i32_base 10 : int
            elif is_float_type x
            then $'backend_switch_has_no_TypeScript_float_rounding_in_lib_spiral' : int
            else $'Number(!x)' : int
        Wasm = fun () =>
            if is_string_type x
            then x |> convert_i32_base 10 : int
            elif is_float_type x
            then $'(i32.trunc_sat_f64_s (f64.nearest !x))' : int
            else x |> numeric_conv : int
        Zig = fun () =>
            if is_string_type x
            then x |> convert_i32_base 10 : int
            elif is_float_type x
            then $'\@as(i32, \@intFromFloat(if (\@abs(!x - \@trunc(!x)) == 0.5) 2.0 * \@round(!x / 2.0) else \@round(!x)))' : int
            else x |> numeric_conv : int
    }

### try_convert

In [ ]:
inl integer_fits forall u. (_ : u) (value : i64) : bool =
    inl bounded : bool = real typecase u with i64 => false | u64 => false | _ => true
    inl lowest : i64 = real typecase u with i8 => -128i64 | i16 => -32768i64 | i32 => -2147483648i64 | _ => 0i64
    inl highest : i64 = real typecase u with i8 => 127i64 | i16 => 32767i64 | i32 => 2147483647i64 | u8 => 255i64 | u16 => 65535i64 | u32 => 4294967295i64 | _ => 0i64
    not bounded || (value >= lowest && value <= highest)

inl parsed_integer forall u. (ok : bool) (value : i64) : option u =
    inl integral : bool = real typecase u with i8 => true | i16 => true | i32 => true | i64 => true | u8 => true | u16 => true | u32 => true | u64 => true | _ => false
    if integral then
        inl converted : u = real conv `i64 `u value
        if ok && integer_fits converted value then Some converted else None
    else None

inl try_convert forall t u. (x : t) : option u =
    x |> to' fun x =>
        backend_switch {
            Gleam = fun () =>
                if (real typecase u with int => true | _ => false) then
                    global "import gleam/int"
                    global "import gleam/string"
                    inl (n : int), (err : int) =
                        $'case int.parse(string.trim(!x)) { Ok(n) -> ##(n, 0) Error(Nil) -> ##(0, 1) }'
                    if err = 0
                    then n |> to |> Some
                    else None : _ u
                else x |> to |> Some
            Lua = fun () =>
                if is_string_type x
                then parsed_integer ($'(string.match(!x, "^%s*[+-]?%d+%s*$") ~= nil)' : bool) ($'(tonumber(!x) or 0)' : i64)
                else x |> to |> Some : _ u
            Fsharp = fun () =>
                resultm.from_try
                    fun () => $'!x |> `u '
                    eval
                |> resultm.ok
                : _ u
            Python = fun () =>
                resultm.from_try
                    fun () => $'`u(!x)'
                    eval
                |> resultm.ok
                : _ u
            Rust = fun () =>
                if (real typecase u with string => true | _ => false)
                then Some $'Rc::<str>::from(format\!("{}", !x))'
                elif (real typecase t with string => true | _ => false)
                then
                    ($'(!x).trim().parse::<`u>().ok()' : optionm'.option' u) |> optionm'.unbox
                else Some $'((!x) as `u)'
            C = fun () =>
                if is_string_type x
                then
                    global "#include <stdbool.h>"
                    global "#include <stdlib.h>"
                    global "#include <ctype.h>"
                    global "#include <errno.h>"
                    global "static bool spiral_integer_text_ok(const char *s) { char *end; while (isspace((unsigned char)*s)) s++; if (!*s) return false; errno = 0; strtoll(s, &end, 10); if (errno != 0 || end == s) return false; while (isspace((unsigned char)*end)) end++; return *end == 0; }"
                    parsed_integer ($'spiral_integer_text_ok((!x)->ptr)' : bool) ($'strtoll((!x)->ptr, NULL, 10)' : i64)
                else x |> to |> Some : _ u
            Delphi = fun () =>
                if is_string_type x
                then parsed_integer ($'(StrToInt64Def(Trim(!x), 0) = StrToInt64Def(Trim(!x), 1))' : bool) ($'StrToInt64Def(Trim(!x), 0)' : i64)
                else x |> to |> Some : _ u
            TypeScript = fun () =>
                if is_string_type x
                then
                    inl ok : bool = $'/^\\s*[+-]?\\d+\\s*$/.test(!x)'
                    parsed_integer ok ($'(!ok ? BigInt(!x) : 0n)' : i64)
                else x |> to |> Some : _ u
            Wasm = fun () =>
                if is_string_type x
                then parsed_integer ($'(call $integer_text_ok !x)' : bool) ($'(call $parse_i64 !x (i32.const 10))' : i64)
                else x |> to |> Some : _ u
            Zig = fun () =>
                if is_string_type x
                then
                    inl ok : bool = $'(if (std.fmt.parseInt(i64, std.mem.trim(u8, !x, " \\t\\r\\n"), 10)) |_| true else |_| false)'
                    parsed_integer ok ($'(std.fmt.parseInt(i64, std.mem.trim(u8, !x, " \\t\\r\\n"), 10) catch 0)' : i64)
                else x |> to |> Some : _ u
        }

In [1]:
///- --test
///> gleam
///> lua
///> fsharp
///> rust

3i32
|> try_convert
|> optionm.value
|> _assert_eq' (
    backend_switch {
        Gleam = fun () => ($'3' : unativeint) : unativeint
        Lua = fun () => ($'3' : unativeint) : unativeint
        Fsharp = fun () => ($'3 |> `unativeint ' : unativeint) : unativeint
        Python = fun () => ($'3' : unativeint) : unativeint
        Rust = fun () => ($'3usize' : unativeint) : unativeint
    } : unativeint
)

.gleam (gleam)
__assert_eq' { expected = 3 }
.lua (lua)
__assert_eq' { expected = 3 }
.fsx (fsharp)
__assert_eq' { expected = 3un }
.rs (rust)
__assert_eq' { expected = 3 }

0

In [2]:
///- --test
///> gleam
///> fsharp
///> rust
///> cuda

"3"
|> try_convert
|> _assert_eq (3i32 |> Some)

"a"
|> try_convert
|> _assert_eq (None : _ int)

true
|> try_convert
|> _assert_eq (1i32 |> Some)

false
|> try_convert
|> _assert_eq (0i32 |> Some)

.gleam (gleam)
__assert_eq { expected = Some(3) }
__assert_eq { expected = None }
__assert_eq { expected = Some(1) }
__assert_eq { expected = Some(0) }
.fsx (fsharp)
__assert_eq { expected = Some(3) }
__assert_eq { expected = None }
__assert_eq { expected = Some(1) }
__assert_eq { expected = Some(0) }
.rs (rust)
__assert_eq { expected = Some(3) }
__assert_eq { expected = None }
__assert_eq { expected = Some(1) }
__assert_eq { expected = Some(0) }
.py (cuda)
__assert_eq { expected = Some(3) }
__assert_eq { expected = None }
__assert_eq { expected = Some(1) }
__assert_eq { expected = Some(0) }

0

### convert

In [ ]:
inl convert forall t u. (x : t) : u =
    x |> try_convert |> optionm.value

### unbox

In [ ]:
inl unbox forall t u. (x : t) : u =
    backend_switch {
        Gleam = fun () => x |> convert : u
        Lua = fun () => x |> to : u
        Fsharp = fun () => $'!x |> unbox<`u>' : u
        Python = fun () => x |> to : u
        Rust = fun () => $'(!x)' : u
        Zig = fun () => x |> to : u
    }

### u8

In [ ]:
inl u8 forall t. (x : t) : u8 =
    backend_switch {
        Gleam = fun () => x |> convert : u8
        Lua = fun () => x |> to : u8
        Fsharp = fun () => x |> $'uint8' : u8
        Python = fun () => x |> to : u8
        Rust = fun () => $'(!x as u8)' : u8
        TypeScript = fun () => $'Number(BigInt.asUintN(8, BigInt(!x)))' : u8
        Zig = fun () => numeric_conv x : u8
        Lean = fun () => numeric_conv x : u8
    }

### u16

In [ ]:
inl u16 forall t. (x : t) : u16 =
    backend_switch {
        Gleam = fun () => x |> convert : u16
        Lua = fun () => x |> to : u16
        Fsharp = fun () => x |> $'uint16' : u16
        Python = fun () => $'!x & 0xFFFF' : u16
        Rust = fun () => $'(!x as u16)' : u16
        TypeScript = fun () => $'Number(BigInt.asUintN(16, BigInt(!x)))' : u16
        Zig = fun () => numeric_conv x : u16
        Lean = fun () => numeric_conv x : u16
    }

### u64

In [ ]:
inl u64 forall t. (x : t) : u64 =
    backend_switch {
        Gleam = fun () => x |> convert : u64
        Lua = fun () => x |> to : u64
        Fsharp = fun () => x |> $'uint64' : u64
        Python = fun () => x |> to : u64
        Rust = fun () => $'(!x as u64)' : u64
        Zig = fun () => numeric_conv x : u64
        Lean = fun () => numeric_conv x : u64
    }

### i32

In [ ]:
inl i32 forall t. (x : t) : i32 =
    backend_switch {
        Gleam = fun () => x |> convert : i32
        Lua = fun () => x |> convert : i32
        Fsharp = fun () => x |> convert : i32
        Python = fun () => x |> convert : i32
        Rust = fun () => x |> to : i32
        Zig = fun () => numeric_conv x : i32
        Lean = fun () => numeric_conv x : i32
    }

### i64

In [ ]:
inl i64 forall t. (x : t) : i64 =
    backend_switch {
        Gleam = fun () => x |> convert : i64
        Lua = fun () => x |> to : i64
        Fsharp = fun () => x |> $'int64' : i64
        Python = fun () => x |> to : i64
        Rust = fun () => x |> to : i64
        Zig = fun () => numeric_conv x : i64
        Lean = fun () => numeric_conv x : i64
    }

### f32

In [ ]:
inl f32 forall t. (x : t) : f32 =
    backend_switch {
        Gleam = fun () => x |> convert : f32
        Lua = fun () => x |> to : f32
        Fsharp = fun () => x |> $'float32' : f32
        Python = fun () => x |> to : f32
        Rust = fun () => $'(!x as f32)' : f32
        Zig = fun () => numeric_conv x : f32
        Lean = fun () => numeric_conv x : f32
    }

### f64

In [ ]:
inl f64 forall t. (x : t) : f64 =
    backend_switch {
        Gleam = fun () => x |> convert : f64
        Lua = fun () => x |> to : f64
        Fsharp = fun () => x |> $'float' : f64
        Python = fun () => x |> to : f64
        Rust = fun () => $'(!x as f64)' : f64
        Zig = fun () => numeric_conv x : f64
        Lean = fun () => numeric_conv x : f64
    }

### to_any

In [ ]:
inl to_any forall t. (obj : t) : any =
    obj |> to

### (~:>) any

In [3]:
instance (~:>) any = to_any